# 02 — Irrigation Models

Build the agronomic irrigation simulation: FAO-56 ET₀ → Kc → ETc → soil-water balance → irrigation requirement.

## 1. Imports and project modules

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import importlib

from src import fao56, crop_calendar

importlib.reload(fao56)
importlib.reload(crop_calendar)


## 2. Load weather data

In [ ]:
DATA_PATH = Path("../data/raw/kiambu_weather.csv")
weather_df = pd.read_csv(DATA_PATH)
weather_df["Date"] = pd.to_datetime(weather_df["Date"])
weather_df["Tmean"] = (
    weather_df["Max_Temperature"] + weather_df["Min_Temperature"]
) / 2


## 3. FAO-56 ET₀ using observed solar radiation

In [ ]:
weather_df["ET0"] = weather_df.apply(
    lambda row: fao56.calculate_daily_et0_from_radiation(
        Tmax=row["Max_Temperature"],
        Tmin=row["Min_Temperature"],
        RH=row["Relative_Humidity"],
        wind_speed=row["Wind_Speed"],
        wind_height=2.0,
        elevation=1700,
        latitude_deg=-1.25,
        day_of_year=row["Date"].dayofyear,
        solar_radiation=row["Solar_Radiation"]
    ),
    axis=1
)

print("Missing ET₀:", weather_df["ET0"].isna().sum())
print("Min:", weather_df["ET0"].min())
print("Max:", weather_df["ET0"].max())
print("Mean:", weather_df["ET0"].mean())


## 4. 120-day annual tomato calendar

In [ ]:
def assign_tomato_calendar(date):
    planting_date = date.replace(month=1, day=1)
    crop_age = (date - planting_date).days

    if crop_age <= 29:
        stage = "Initial"
    elif crop_age <= 59:
        stage = "Development"
    elif crop_age <= 89:
        stage = "Mid-season"
    elif crop_age <= 119:
        stage = "Late-season"
    else:
        stage = "No crop"

    return planting_date, crop_age, stage

weather_df[["planting_date","crop_age","growth_stage"]] = weather_df["Date"].apply(
    lambda x: pd.Series(assign_tomato_calendar(x))
)
weather_df["Kc"] = weather_df["growth_stage"].apply(crop_calendar.get_kc)

display(weather_df["growth_stage"].value_counts())


## 5. Crop evapotranspiration

In [ ]:
weather_df["ETc"] = weather_df["ET0"] * weather_df["Kc"]
display(weather_df[["Date","ET0","Kc","ETc","growth_stage"]].head(20))


## 6. Soil parameters → TAW and RAW

In [ ]:
weather_df["field_capacity"] = 0.30
weather_df["wilting_point"] = 0.15
weather_df["root_depth"] = 0.60

weather_df["TAW"] = (
    (weather_df["field_capacity"] - weather_df["wilting_point"])
    * weather_df["root_depth"] * 1000
)
weather_df["RAW"] = 0.40 * weather_df["TAW"]

print("TAW:", weather_df["TAW"].iloc[0], "mm")
print("RAW:", weather_df["RAW"].iloc[0], "mm")


## 7. Effective rainfall

In [ ]:
weather_df["effective_rainfall"] = weather_df["Rainfall"] * 0.80


## 8. Soil-water balance and irrigation

In [ ]:
depletion = 0.0
before, after, decision, net = [], [], [], []

for _, row in weather_df.iterrows():
    d_pre = max(
        0.0,
        min(
            row["TAW"],
            depletion + row["ETc"] - row["effective_rainfall"]
        )
    )

    if row["Kc"] == 0:
        irrigate = False
        net_irrigation = 0.0
        depletion = 0.0
    elif d_pre >= row["RAW"]:
        irrigate = True
        net_irrigation = d_pre
        depletion = 0.0
    else:
        irrigate = False
        net_irrigation = 0.0
        depletion = d_pre

    before.append(d_pre)
    after.append(depletion)
    decision.append(irrigate)
    net.append(net_irrigation)

weather_df["depletion_before_irrigation"] = before
weather_df["depletion"] = after
weather_df["irrigate"] = decision
weather_df["net_irrigation"] = net
weather_df["irrigation_efficiency"] = 0.80
weather_df["gross_irrigation"] = (
    weather_df["net_irrigation"] / weather_df["irrigation_efficiency"]
)


## 9. Irrigation results

In [ ]:
print("Irrigation events:", int(weather_df["irrigate"].sum()))
print("Total net irrigation:", weather_df["net_irrigation"].sum(), "mm")
print("Total gross irrigation:", weather_df["gross_irrigation"].sum(), "mm")

display(
    weather_df.loc[
        weather_df["irrigate"],
        ["Date","growth_stage","ETc","effective_rainfall",
         "depletion_before_irrigation","RAW","net_irrigation","gross_irrigation"]
    ].head(20)
)


## 10. Save processed dataset

In [ ]:
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

weather_df.to_csv(OUT / "irrigation_dataset.csv", index=False)
print("Saved:", OUT / "irrigation_dataset.csv")


## 11. Interpretation

This notebook establishes the agronomic baseline used to create the irrigation targets for machine learning.